# BG RL — آموزش تخته‌نرد در Google Colab

این notebook برای آموزش headless است و رابط pygame را روی Windows اجرا می‌کنیم.

برای این پروژه‌ی کوچک، پیش‌فرض notebook روی CPU است؛ چون تولید حرکت‌های قانونی تخته‌نرد با Python/CPU انجام می‌شود و GPU ضعیف ممکن است مزیت زیادی نداشته باشد. اگر GPU سریع و پایدار بود، فقط `DEVICE = 'auto'` را تغییر دهید.

checkpointها در Google Drive ذخیره می‌شوند و آموزش به chunkهای کوتاه تقسیم شده است تا بعد از قطع Colab بتوان آن را ادامه داد.

In [ ]:
# تنظیمات repository
REPO_URL = 'https://github.com/Mooli-web/BG.git'
BRANCH = 'arena/01a0b49e-bg'
REPO_DIR = '/content/BG'

import os
import shutil
import subprocess
import sys

if os.path.exists(REPO_DIR) and not os.path.exists(os.path.join(REPO_DIR, '.git')):
    shutil.rmtree(REPO_DIR)
if not os.path.exists(os.path.join(REPO_DIR, '.git')):
    subprocess.run(['git', 'clone', '-b', BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'pull', 'origin', BRANCH], check=True)

%cd /content/BG
print('Repository ready:', os.getcwd())

In [ ]:
# نصب وابستگی‌های آموزش؛ pygame برای Colab لازم نیست.
%pip install -q -r requirements-colab.txt

import numpy as np
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

# برای پایداری، CPU انتخاب شده است. برای آزمایش GPU مقدار را به 'auto' تغییر دهید.
DEVICE = 'cpu'

In [ ]:
# اتصال Google Drive برای نگه‌داری checkpointها
from google.colab import drive
drive.mount('/content/drive')

CHECKPOINT_DIR = '/content/drive/MyDrive/BG_RL/checkpoints_fixed_rules'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print('Checkpoints:', CHECKPOINT_DIR)

## آموزش chunk به chunk

در هر اجرا حداکثر `CHUNK_STEPS` تصمیم آموزش داده می‌شود. اگر Colab قطع شود، همین سلول را دوباره اجرا کنید؛ آخرین `latest.pt` از Drive خوانده می‌شود و ادامه می‌دهد.

برای شروع از صفر از یک پوشه‌ی جدید استفاده کنید و `--resume` را دستی اضافه نکنید. مدل‌های قبل از اصلاح قانون bearing off را استفاده نکنید.

In [ ]:
import os
import subprocess
import sys
import torch

TARGET_STEPS = 20_000_000
CHUNK_STEPS = 250_000
NUM_ENVS = 8
ROLLOUT_STEPS = 128
SAVE_INTERVAL = 5

latest_path = os.path.join(CHECKPOINT_DIR, 'latest.pt')

def saved_steps(path):
    if not os.path.exists(path):
        return 0
    checkpoint = torch.load(path, map_location='cpu')
    return int(checkpoint.get('global_steps', checkpoint.get('total_steps', 0)))

current_steps = saved_steps(latest_path)
while current_steps < TARGET_STEPS:
    next_target = min(current_steps + CHUNK_STEPS, TARGET_STEPS)
    command = [
        sys.executable, '-m', 'bg', 'train',
        '--algorithm', 'td_lambda',
        '--total-steps', str(next_target),
        '--num-envs', str(NUM_ENVS),
        '--rollout-steps', str(ROLLOUT_STEPS),
        '--device', DEVICE,
        '--checkpoint-dir', CHECKPOINT_DIR,
        '--save-interval', str(SAVE_INTERVAL),
        '--log-interval', '10',
        '--seed', '7',
        '--reward-shaping', '0.0005',
        '--td-lambda', '0.70',
        '--search-samples', '1',
    ]
    if current_steps > 0:
        command += ['--resume', latest_path]
    print(f'Running from {current_steps:,} to {next_target:,} steps')
    subprocess.run(command, check=True)
    current_steps = saved_steps(latest_path)
print(f'Target reached: {current_steps:,} steps')

## ادامه تا ۲۰ میلیون step

مقدار پیش‌فرض `TARGET_STEPS` برابر `20_000_000` است. اگر فعلاً مدل کوچک‌تری می‌خواهید، آن را به `5_000_000` یا `10_000_000` تغییر دهید. اگر runtime قطع شد، همان سلول آموزش را دوباره اجرا کنید.

In [ ]:
# ارزیابی متنی در Colab، بدون pygame
evaluate_command = [
    sys.executable, '-m', 'bg', 'evaluate',
    '--checkpoint', os.path.join(CHECKPOINT_DIR, 'latest.pt'),
    '--games', '100',
    '--device', DEVICE,
]
subprocess.run(evaluate_command, check=True)

In [ ]:
# دانلود checkpoint برای بازی روی Windows
from google.colab import files
files.download(os.path.join(CHECKPOINT_DIR, 'latest.pt'))